## Update propagation — env vars vs volume mounts

The big practical difference between the consumption patterns is **what happens when you update the ConfigMap.**

### Env vars are baked in

Both `configMapKeyRef` and `envFrom` are resolved **once, at pod start.** The kubelet reads the ConfigMap, builds the env block, and `exec`s the container. After that, changing the ConfigMap does **nothing** to running pods. To pick up the new value you restart the pod — typically by triggering a rollout (notebook 03: bump a template annotation).

### Volume mounts live-update

A ConfigMap mounted as a volume is written to a `tmpfs` dir inside the pod. When it changes, the kubelet **rewrites those files** within about a minute (the sync period). The container sees new contents **without restarting.**

That's great — nginx reloads on `kubectl apply` — and a footgun: the container must actually **re-read** the file. Many apps load config once at start, so the file is fresh but the app still uses the old values. Fixes: **have the app watch the file** (inotify / periodic re-read), or **trigger a rollout on every change** (a checksum annotation in the pod template — the GitOps pattern).

### The `subPath` gotcha

A volume mounted with **`subPath` is *not* live-updated.** `subPath` lands a single file (e.g. only `nginx.conf` into `/etc/nginx/`) as a separate bind mount the kubelet doesn't refresh. If you need live updates *and* a single file in an un-replaceable dir, mount the whole ConfigMap somewhere harmless and read from there.

On our map this is the **env / mount** chip: the *same* ConfigMap behaves differently depending on which projection you chose — env = frozen at start, volume = living.